# Lesson 21: Capstone projects and your AI engineering career

*Worked solutions for [the lesson page](../../lessons/21-capstones-career.html). The exercise version without solutions is `../21-capstones-career.ipynb`.*

Prove your skills by building one end-to-end system from five capstone specifications with rubrics, then turn the work into a portfolio, prepare for interviews with worked questions, and plan your next ninety days.

**Runs on:** Any computer. This notebook is a capstone workbook.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy

## The seven capstones

Read the seven specifications and the rubric in the lesson, then scope yours here.

### Hands-on: scope your capstone

Write a one-page project brief before writing any code:

1. One sentence: who it is for and what it does.
2. Three measurable success criteria.
3. Your data sources and how you will build the eval set.
4. The simplest architecture that could work, as a sketch.
5. The three riskiest assumptions, and how you will test each in the first week.
6. What you will deliberately **not** build.

#### Solution: an example brief (Capstone A)

**Who and what**: answers calibration and maintenance questions for one instrument model from its 420-page service manual and 60 service bulletins, with page citations, for field technicians.

**Success criteria**: (1) correct page in the top 6 retrieved for at least 90% of 60 answerable test questions; (2) faithful, correctly cited answers for at least 85% as judged by a calibrated judge; (3) says "not in the documents" for at least 9 of 10 unanswerable questions; median time to first token under 2 seconds.

**Data and eval**: the PDF manual and bulletins; 40 questions from two experienced technicians plus 20 reviewed synthetic questions and 10 unanswerable ones; gold page numbers recorded for each.

**Architecture**: Docling parsing, section-level chunks with title and section path, BM25 plus embeddings with RRF, cross-encoder reranking to 6, a mid-tier model with the grounded prompt, FastAPI streaming, JSON-line traces.

**Riskiest assumptions**: tables of torque values parse correctly (test 10 tables on day one); technicians phrase questions very differently from the manual (collect real questions early); bulletins that supersede manual sections can be detected (check metadata).

**Not building**: user accounts, a mobile app, multi-manual support, fine-tuning.

## Build the walking skeleton

One entry point, configuration in one place, traces from day one. The harness below ties your pipeline to your eval cases; you will run it hundreds of times.

In [ ]:
import json
import time


def run_eval(pipeline, grade, cases, trace_path="traces.jsonl"):
    """Run every case through the pipeline, trace it, return the pass rate."""
    passed = 0
    with open(trace_path, "a") as f:
        for case in cases:
            t0 = time.perf_counter()
            result = pipeline(case["input"])
            ok = grade(result, case["expected"])
            passed += ok
            f.write(json.dumps({"id": case["id"], "input": case["input"],
                                "result": result, "passed": ok,
                                "seconds": round(time.perf_counter() - t0, 4)}) + "\n")
    return passed / len(cases)

### Hands-on: a traced walking skeleton

Build the skeleton for your capstone here, with placeholders where the real components will go.

1. Write at least ten eval cases in the format below and save them as `eval_cases.jsonl`. Include two that should be refused or abstained on.
2. Write `pipeline(text)` with every stage present as a function, using the simplest logic you can: keyword matching for retrieval, a fixed template for generation, rules for detection. Every stage returns something that goes into the result dictionary.
3. Write `grade(result, expected)`, a code grader that returns 1 or 0.
4. Run `run_eval` and read `traces.jsonl`.

You are done when every stage appears in every trace line and the harness prints a pass rate, however low.

In [ ]:
cases = [
    {"id": "q-001", "input": "Replace with a real question",
     "expected": "what a correct answer must contain", "tags": ["lookup"]},
    {"id": "q-002", "input": "A question your data cannot answer",
     "expected": "NOT_IN_DOCS", "tags": ["unanswerable"]},
]
with open("eval_cases.jsonl", "w") as f:
    for c in cases:
        f.write(json.dumps(c) + "\n")
print(len(cases), "cases written")

#### Solution

A skeleton for Capstone A over a six-paragraph toy corpus: `retrieve` counts query words found in each paragraph, `generate` returns the best paragraph with its id as the citation, and `grade` checks that the expected phrase is in the answer. Crude on purpose: the point is that every stage exists and every run is traced.

In [ ]:
# A skeleton for Capstone A over a toy corpus. Every stage is present and crude.
DOCS = {
    "p-01": "Calibration: run the zero check with the shutter closed, then the span "
            "check with the reference lamp. Repeat the calibration every 30 days.",
    "p-02": "Battery: the internal battery lasts 8 hours. Replace the battery only "
            "with part number BT-220.",
    "p-03": "Error codes: E14 means the detector is too warm; move the instrument "
            "to shade and wait 10 minutes. E21 means the lamp has failed.",
    "p-04": "Cleaning: wipe the optical window with lens tissue and isopropyl "
            "alcohol. Never use acetone on the window.",
    "p-05": "Firmware: download the latest firmware from the support site and "
            "install it over USB. Installation takes about 4 minutes.",
    "p-06": "Storage: store the instrument between 5 and 40 degrees with the lens "
            "cap on. Remove the battery for storage longer than a month.",
}

cases = [
    {"id": "q-001", "input": "How often should I run the calibration?", "expected": "every 30 days", "tags": ["lookup"]},
    {"id": "q-002", "input": "What does error E14 mean?", "expected": "too warm", "tags": ["lookup"]},
    {"id": "q-003", "input": "Which battery part number do I need?", "expected": "BT-220", "tags": ["lookup"]},
    {"id": "q-004", "input": "Is it OK to clean the window with acetone?", "expected": "never use acetone", "tags": ["paraphrase"]},
    {"id": "q-005", "input": "How long does installing new firmware take?", "expected": "4 minutes", "tags": ["paraphrase"]},
    {"id": "q-006", "input": "What temperatures are safe when storing it?", "expected": "5 and 40 degrees", "tags": ["paraphrase"]},
    {"id": "q-007", "input": "The lamp has failed. Which error code shows?", "expected": "E21", "tags": ["lookup"]},
    {"id": "q-008", "input": "How many hours does the battery last?", "expected": "8 hours", "tags": ["lookup"]},
    {"id": "q-009", "input": "Does the instrument have Wi-Fi?", "expected": "NOT_IN_DOCS", "tags": ["unanswerable"]},
    {"id": "q-010", "input": "What is the warranty period?", "expected": "NOT_IN_DOCS", "tags": ["unanswerable"]},
]
with open("eval_cases.jsonl", "w") as f:
    for c in cases:
        f.write(json.dumps(c) + "\n")
print(len(cases), "cases written")


def retrieve(query, k=1):
    """Stage 1, crude: count query words that appear in each paragraph."""
    words = query.lower().split()
    scored = sorted(DOCS, key=lambda d: -sum(w in DOCS[d].lower() for w in words))
    return scored[:k]


def generate(query, doc_ids):
    """Stage 2, crude: return the best paragraph as the answer, with a citation."""
    if not doc_ids:
        return {"answer": "NOT_IN_DOCS", "citation": None}
    return {"answer": DOCS[doc_ids[0]], "citation": doc_ids[0]}


def pipeline(text):
    docs = retrieve(text)
    out = generate(text, docs)
    return {"retrieved": docs, **out}


def grade(result, expected):
    if expected == "NOT_IN_DOCS":
        return int(result["answer"] == "NOT_IN_DOCS")
    return int(expected.lower() in result["answer"].lower())


rate = run_eval(pipeline, grade, cases, "traces.jsonl")
print("pass rate:", rate)
with open("traces.jsonl") as f:
    traces = {json.loads(line)["id"]: json.loads(line) for line in f}
print("worst traces:", " ".join(t["id"] for t in traces.values() if not t["passed"]))

Open the failed traces: retrieval, not generation, is the weak stage. A paraphrased question (q-006) picks the wrong paragraph, and the two unanswerable ones (q-009, q-010) are never refused, because counting raw words (including *the*) always finds some paragraph to return. That is the first decision-log entry and the first thing to improve in the next session.

## Evaluate, harden and deploy

### One change at a time, with an interval

Two pass rates are not enough: with 50 cases, a change of three cases is noise. Put an interval on the difference and list the fixed and broken cases.

In [ ]:
import numpy as np


def paired_bootstrap(before, after, n=5000, seed=0):
    """Mean change and a 95% interval for the change in pass rate between two
    runs on the same cases."""
    rng = np.random.default_rng(seed)
    before, after = np.asarray(before, float), np.asarray(after, float)
    idx = rng.integers(0, len(before), size=(n, len(before)))
    diffs = after[idx].mean(axis=1) - before[idx].mean(axis=1)
    return (after - before).mean(), np.percentile(diffs, [2.5, 97.5])

### Hands-on: one measured improvement

1. Make one real change to the weakest stage of your skeleton (for Capstone A, replace keyword counting with ranking by the fraction of query words matched, or with embeddings from the embeddings and vector search lesson).
2. Run the harness again into a second trace file.
3. From the two trace files, list the fixed and broken case ids and compute `paired_bootstrap` on the per-case results.
4. Write the decision-log row: date, decision, alternatives, evidence (the interval), result.

You are done when you have a decision-log row with an interval in it, and can say whether the change is real or noise.

#### Solution

Rank paragraphs by the fraction of query tokens matched after lower-casing, stripping punctuation and dropping stopwords, and abstain when even the best paragraph matches less than half of them. Then read both trace files back and compare per case.

In [ ]:
import re

STOPWORDS = {"the", "a", "an", "is", "it", "to", "i", "do", "does", "of", "with",
             "what", "which", "how", "when", "should", "ok", "my", "and", "new"}


def tokens(text):
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS}


def retrieve_v2(query, k=1, min_fraction=0.5):
    """Fraction of (non-stopword) query tokens present in each paragraph;
    abstain when even the best paragraph matches less than half of them."""
    q = tokens(query)
    scores = {d: len(q & tokens(DOCS[d])) / max(len(q), 1) for d in DOCS}
    best = sorted(scores, key=lambda d: -scores[d])[:k]
    return [d for d in best if scores[d] >= min_fraction]


def pipeline_v2(text):
    docs = retrieve_v2(text)
    return {"retrieved": docs, **generate(text, docs)}


rate_v2 = run_eval(pipeline_v2, grade, cases, "traces_v2.jsonl")


def results(path):
    with open(path) as f:
        rows = [json.loads(line) for line in f]
    return {r["id"]: r["passed"] for r in rows}        # last run per id wins


before, after = results("traces.jsonl"), results("traces_v2.jsonl")
ids = [c["id"] for c in cases]
b = [before[i] for i in ids]
a = [after[i] for i in ids]
fixed = [i for i in ids if not before[i] and after[i]]
broke = [i for i in ids if before[i] and not after[i]]
change, (lo, hi) = paired_bootstrap(b, a)
print(f"before {sum(b) / len(b):.1f}  after {sum(a) / len(a):.1f}")
print(f"fixed: {fixed}  broke: {broke}")
print(f"change {change:+.2f}  95% interval [{lo:.1f}, {hi:.1f}]")

The change fixed both unanswerable cases (the skeleton now abstains) and broke one paraphrase: "installing" no longer matches "install" once whole words are compared. The fixed and broken lists point at the next change (stemming, or embeddings), and the interval says the evidence is thin.

**Decision-log row.** 2026-10-19 | Rank paragraphs by the fraction of query tokens matched, abstain below 0.5 | Alternatives: keep keyword counts; embeddings | Evidence: 0.7 to 0.8 on ten cases, interval [-0.2, 0.4]; fixed q-009 and q-010, broke q-005 | Result: kept; next, add stemming and thirty more cases before touching the model.

## Write it up

### Hands-on: a results-first README

Draft the first screen of your README, the part a reader sees without scrolling:

1. The problem in two sentences, naming the user.
2. A results table: baseline versus final on your two or three headline metrics, with the interval or sample size, plus cost per thousand requests and p50 latency.
3. Three bullet points of "what I tried", each with the evidence and including one thing that did not work.
4. The five commands to run it.

You are done when a colleague who has never seen the project can say in one minute what it does, how well, and how to try it.

#### Solution: an example first screen (Capstone F)

**Problem**: a 3B model extracts structured fields from maintenance tickets for a fleet of 200 instruments. Technicians need results in under a second, and the team wants to serve it on one small GPU.

| Configuration | Field accuracy (n=120) | p50 / p95 latency | Cost per 1,000 requests |
|---|---|---|---|
| Hosted API (baseline) | 0.93 | 1.4 s / 2.9 s | measured, in the report |
| Self-hosted 16-bit, no batching | 0.91 | 0.9 s / 3.1 s | measured |
| Self-hosted 4-bit, batched, prefix cache | 0.90 | 0.4 s / 0.8 s | measured |

**What I tried**: 4-bit weights cost one point of accuracy and halved memory (interval on the difference included zero, so treated as a tie); speculative decoding with a 0.5B draft gave a 1.6 times decode speed-up at an acceptance rate of about 70%, but fell to no gain on short outputs, so it is off by default; constrained JSON decoding removed every schema error but added 15% to latency.

**Run it**: clone, `docker compose up`, `make eval`, `make load-test`, open the dashboard. The real README has the measured costs in the table; never publish a results table with placeholders.

## Interview preparation

**Practice question:** Explain the bias-variance trade-off and how you diagnose it.

Bias is error from overly simple assumptions (underfitting: high training and validation error); variance is error from sensitivity to the particular training data (overfitting: low training error, high validation error). Diagnose by comparing training and validation error; fix underfitting with more capacity or better features, overfitting with more data, regularisation, simpler models or early stopping (ML fundamentals lesson).

**Practice question:** Walk through scaled dot-product attention. Why divide by √d_k?

Each token's query is dotted with every key, scaled, softmaxed into weights, and used to average the values: softmax(QKᵀ/√d_k)V. Dot products of random d-dimensional vectors have standard deviation √d, so without scaling large dimensions push softmax into saturation and gradients vanish (transformers lesson).

**Practice question:** What is the KV cache, and why does it matter for serving?

During generation, keys and values of previous tokens are stored so each step only computes the new token's projections. It makes generation efficient but consumes memory proportional to layers × KV heads × head dimension × context length × concurrent sequences, which often limits how many users a GPU can serve; paged allocation and grouped-query attention exist to shrink it (transformers and LLM inference lessons).

**Practice question:** How do quantisation, batching and speculative decoding make serving cheaper?

Decoding one token reads every weight once, so it is bound by memory bandwidth, not arithmetic. Quantisation stores weights in 8 or 4 bits, cutting memory and the bytes read per token, at a small and measurable quality cost. Continuous batching serves many sequences per weight read, raising throughput without hurting each request much. Speculative decoding lets a small draft model propose several tokens that the large model verifies in one parallel pass, producing the same distribution faster when the acceptance rate is high (LLM inference lesson).

**Practice question:** When would you use a reasoning model, and what does it cost?

Reasoning models spend extra tokens thinking before they answer, which helps on multi-step problems: maths, code, planning, diagnosis across several pieces of evidence. The cost is latency and output tokens, controlled by a thinking budget. They are not better at everything, so route only the hard cases to them, evaluate on your own set, and keep extraction and lookup on a standard model (reasoning models lesson).

**Practice question:** What is context engineering, and how does memory work in an agent?

Context engineering is deciding what enters the model's window on each turn: instructions, tools, retrieved documents, recent history and memory, within a token budget. Long conversations are compacted into rolling summaries; durable facts go to an external memory store and are retrieved when relevant rather than carried forever; stable prefixes are ordered first so that prompt caching applies (context engineering lesson).

**Practice question:** How would you reduce hallucinations in a customer-facing assistant?

Ground answers in retrieved sources with citations; instruct and allow abstention; use structured outputs and tools for facts and arithmetic; lower temperature for factual tasks; add a faithfulness check step for high-risk answers; evaluate faithfulness and abstention on a golden set and monitor them in production (RAG and evaluation lessons).

**Practice question:** RAG versus fine-tuning versus long context: how do you choose?

RAG for knowledge that is large, changing, access-controlled or needs citations. Fine-tuning for behaviour, format, style, or making a small model do a narrow task cheaply. Long context with caching for small, stable collections where simplicity matters. Often combined (RAG, fine-tuning and context engineering lessons).

**Practice question:** Explain LoRA. Why does it work, and what does it save?

Freeze the pretrained weights and learn a low-rank update BA (rank r much smaller than the layer dimensions), scaled by α/r, with B initialised to zero. Weight updates during fine-tuning have low intrinsic rank, so this captures most of the benefit while training under 1% of the parameters, cutting optimiser memory dramatically and producing small swappable adapters; QLoRA also stores the base model in 4-bit (fine-tuning lesson).

**Practice question:** How do you evaluate an LLM feature when outputs are open-ended?

A golden dataset from real usage with hard and negative cases; code graders for anything rule-based; LLM judges on specific criteria with clear rubrics, structured verdicts, order swapping for pairwise comparisons, and calibration against human grades; per-slice reporting with confidence intervals and fixed/broken case analysis; online signals in production (evaluation lesson).

**Practice question:** What is prompt injection and how do you defend against it?

Instructions embedded in content the model reads (direct from users, or indirect via documents, web pages, tool results) that try to override the system's instructions. No complete fix exists, so defend in depth: separate and label untrusted content, but above all limit impact with least-privilege tools, human approval for consequential actions, output filtering, safe rendering and avoiding agents that combine untrusted input, private data and external communication (safety and security lesson).

**Practice question:** Explain RLHF, DPO and the RL methods behind reasoning models.

RLHF: collect human preference comparisons, train a reward model to predict them, then optimise the language model with reinforcement learning (PPO) to maximise reward with a KL penalty that keeps it close to the supervised model. DPO: skip the reward model and RL, and optimise the preference pairs directly with a classification-style loss relative to a frozen reference model; simpler and more stable. Reasoning models are trained with RL against verifiable rewards (a test passes, an answer matches), with group-based methods such as GRPO that drop the separate value model (how LLMs are built, advanced fine-tuning and reinforcement learning lessons).

**Practice question:** When is a language model the wrong tool?

When the task is numbers, structure or control at volume: tabular prediction (gradient boosting), forecasting (seasonal baselines and boosting on lag features), ranking for users (collaborative filtering), images at scale (a trained CNN or vision transformer), routes and schedules (search and constraint solvers), actions with delayed reward (reinforcement learning). The language model then does the language around it: parsing requests, explaining results, writing reports (system design lesson and part IX).

**Practice question:** How do you evaluate a recommender?

Offline: a time-based split, recall@k and NDCG@k against the held-out interactions, always against a popularity baseline, with coverage to catch a model that only recommends hits. Offline metrics are biased by what the old system showed people, so the decision is made online: an A/B test with a primary metric, guardrail metrics and a sample-size estimate, watching for feedback loops (time series, recommenders and graphs lesson).

**Practice question:** Your RAG system's answers got worse after a change. How do you debug it?

Reproduce on the eval set and list newly broken cases. For each, check the trace: was the right chunk retrieved (parsing, chunking, embedding, filters)? Was it ranked high enough (reranker)? Was it in the prompt but ignored or contradicted (prompt, model, context length)? Fix the stage responsible, re-run the full eval, and add the cases to the regression set (RAG, evaluation and deployment lessons).

### Coding interviews

Practise against a timer, without libraries beyond NumPy.

### Hands-on: a timed coding drill

Set a timer for twenty minutes and, in NumPy only, implement:

1. `top_k_cosine(query, matrix, k)`: the indices of the k rows of `matrix` most similar to `query` by cosine similarity, most similar first.
2. `causal_attention(Q, K, V)`: scaled dot-product attention with a causal mask, for a single head with shapes (n, d).

Then write three asserts for each: a hand-checkable case, a shape check and an edge case (a zero vector; n equal to 1). You are done when all six asserts pass inside the time limit, and you can explain both functions aloud without looking.

#### Solution

Normalise rows and query, dot, and `argsort` the negated similarities with a stable sort so that ties keep their order. For attention, mask above the diagonal with minus infinity before the softmax and subtract the row maximum for stability.

In [ ]:
import numpy as np


def top_k_cosine(query, matrix, k):
    q = query / (np.linalg.norm(query) + 1e-12)
    m = matrix / (np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-12)
    sims = m @ q
    return np.argsort(-sims, kind="stable")[:k]


def causal_attention(Q, K, V):
    n, d = Q.shape
    scores = Q @ K.T / np.sqrt(d)
    mask = np.triu(np.ones((n, n), dtype=bool), k=1)   # True above the diagonal
    scores = np.where(mask, -np.inf, scores)
    scores = scores - scores.max(axis=1, keepdims=True)  # stable softmax
    weights = np.exp(scores)
    weights = weights / weights.sum(axis=1, keepdims=True)
    return weights @ V, weights


# top-k: hand-checkable, shape, edge case
M = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [-1.0, 0.0], [2.0, 1.0]])
top = top_k_cosine(np.array([1.0, 1.0]), M, 3)
print("top-3 for the query:", top)
assert list(top) == [2, 4, 0]                 # [1,1] first, then [2,1], then [1,0] (tie, stable)
assert top_k_cosine(np.array([1.0, 1.0]), M, 2).shape == (2,)
assert not np.isnan(top_k_cosine(np.zeros(2), M, 5)).any()   # zero query: no NaN

# attention: rows sum to one, causality, n = 1
rng = np.random.default_rng(0)
Q, K, V = (rng.normal(size=(4, 8)) for _ in range(3))
out, w = causal_attention(Q, K, V)
print("attention row sums:", w.sum(axis=1).round(6))
print("first row attends only to itself:", bool(w[0, 0] == 1.0 and w[0, 1:].sum() == 0))
assert np.allclose(w.sum(axis=1), 1.0)
assert np.allclose(np.triu(w, k=1), 0.0) and out.shape == (4, 8)
out1, w1 = causal_attention(Q[:1], K[:1], V[:1])
assert np.allclose(out1, V[:1]) and w1.shape == (1, 1)
print("all asserts passed")

## Career paths and the next ninety days

### Hands-on: your ninety-day plan

Write a plan with three 30-day blocks. For each block: one thing to build, one topic to go deeper on, one thing to publish, and one way to get feedback from people more experienced than you. Put it at the top of your learning log, and schedule a review at day 30.

#### Solution: an example plan

**Days 1 to 30**: build Capstone A on the service manuals you use at work (with permission, or on public equivalents); go deeper on retrieval evaluation; publish the repository with a results table; get feedback by demoing to two colleagues who would actually use it.

**Days 31 to 60**: extend it with an MCP server and an agent mode (Capstone B ideas); go deeper on security and prompt injection; publish a write-up of what worked and what did not; get feedback by asking a security-minded colleague to attack it.

**Days 61 to 90**: distil the most common query type into a fine-tuned small model and serve it quantised and batched (Capstones D and F); go deeper on inference optimisation; publish a cost-and-quality comparison; get feedback through an open-source contribution or a talk at a local meetup.

## Capstone workbook

Keep this notebook (or a copy of it) at the root of your capstone repository. The eval-case writer and the harness above are its first two tools; add your decision log and results table here.

### Decision log

| Date | Decision | Alternatives considered | Evidence | Result |
|---|---|---|---|---|
| | | | | |

### Results table for the write-up

| System | Metric 1 | Metric 2 | Cost per 1,000 requests | p50 latency |
|---|---|---|---|---|
| Baseline | | | | |
| Final | | | | |

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/32-capstones-career.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.